# Multi-agent Personal Assistant System Built on Supervisor Pattern

_Building a simple multi-agent personal assistant system following a supervisor/manager and sub-agents pattern._

A supervisor agent coordinates the following two specialized workers or sub-agents to complete a relatively complex task. Each sub-agent completes its task, and the supervisor synthesizes both results into a coherent response to share with the user.

A Calender Agent: It handles availability check, schedulning and event management.

An Email Agent: It drafts messages, manages communication and send notifications.

In [1]:
# Imports packages

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
from datetime import datetime

## Model

_Connecting an appropriate model to a chat client._

In [2]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "qwen3.5:2b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [3]:
# Initializes chat client
model = ChatOllama(
    model=OLLAMA_MODEL,
    reasoning=False,
    base_url=OLLAMA_ENDPOINT)

## Tools

Defines all the tools by the system.

In [4]:
@tool
def create_calender_event(
    title: str,
    start_time: str,        # ISO format: "2024-01-15T14:00:00"
    end_time: str,          # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],   # Email addresses
    location: str,    
) -> str:
    """Creates a calender event. Requires datetime in ISO format."""

    # A dummy function for this experiment. In practice, this would call Google Calendar API, Outlook API, etc.

    return f"Event was created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

In [5]:
@tool
def send_email(
    to: list[str],          # Email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""

    # A dummy function for this experiment. In practice, this would call SendGrid, Gmail API, etc.

    return f"Email sent to {', '.join(to)} - Subject: {subject}"

In [6]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,              # ISO format: "2024-01-15"
    duration_minutes: int,
) -> list[str]:
    """Check calendar availability for given attendees on a specific date. Requires date in ISO format."""

    # A dummy function for this experiment. In practice, this would query calendar APIs

    return ["09:00", "14:00", "16:00"]

## Sub-Agents
_Creating specialized sub-agents._

### Calendar Agent
The calendar agent understands natural language scheduling requests and translates them into precise API calls. It handles date parsing, availability checking, and event creation.

In [7]:
CALENDAR_AGENT_PROMPT = f"""
    You are a calendar scheduling assistant.
    Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm')
    into proper ISO datetime formats. Today is {datetime.date(datetime.now()).strftime("%A")}.
    Current time is {datetime.now().isoformat()} (in ISO format).
    Use 'get_available_time_slots' to check availability when needed.
    If there is no suitable time slot, stop and confirm unavailability in your response.
    Use 'create_calendar_event' to schedule events.
    Always confirm what was scheduled in your final response.
"""

# Creates the calendar agent
calendar_agent = create_agent(
    model,
    tools=[create_calender_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    )

In [8]:
# Tests the calendar agent to check how it handles event scheduling in natural language
# MAY TAKE NEARLY A MINUTE TO COMPLETE

query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

agent_output = calendar_agent.invoke(
    {"messages": [{"role": "user", "content": query}]}, 
    stream_mode="values",   # Instructs to receive output values instead of events
    version="v2"            # In version v2, it returns last output of the run
    )

# Prints the messages for analysis
for message in agent_output.value["messages"]:
    message.pretty_print()

================================ Human Message =================================

Schedule a team meeting next Tuesday at 2pm for 1 hour
================================== Ai Message ==================================
Tool Calls:
  get_available_time_slots (f3308e8b-03ab-4c96-864d-56c00e4994d7)
 Call ID: f3308e8b-03ab-4c96-864d-56c00e4994d7
  Args:
    attendees: ['team']
    date: 2026-06-23T14:00:00
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calender_event (dce312e4-db9a-4b85-a2f2-a83f1fcdd50e)
 Call ID: dce312e4-db9a-4b85-a2f2-a83f1fcdd50e
  Args:
    title: Team Meeting
    start_time: 2026-06-23T14:00:00
    end_time: 2026-06-23T15:00:00
    attendees: ['team']
    location: Conference Room A
================================= Tool Message =====================

### Email Agent
The email agent handles message composition and sending. It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [9]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use 'send_email' to send the message. "
    "Always confirm what was sent in your final response."
)

# Creates email agent
email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

In [10]:
# Tests the email agent to check how it handles email related instructions in natural language
# MAY TAKE NEARLY A MINUTE TO COMPLETE

query = "Send the quality assurance team a reminder about reviewing the new test cases."

agent_output = email_agent.invoke(
    {"messages": [{"role": "user", "content": query}]}, 
    stream_mode="values", 
    version="v2"
    )

for message in agent_output.value["messages"]:
    message.pretty_print()

================================ Human Message =================================

Send the quality assurance team a reminder about reviewing the new test cases.
================================== Ai Message ==================================
Tool Calls:
  send_email (3d6a0f02-ebeb-4a26-bf72-5b7adf9c7766)
 Call ID: 3d6a0f02-ebeb-4a26-bf72-5b7adf9c7766
  Args:
    to: ['qa@company.com']
    subject: Reminder: Reviewing New Test Cases
    body: Hi QA Team,

This is a quick reminder to start reviewing the new test cases. Please check the attached documents and provide any updates or feedback by [insert deadline].

Let's keep the quality assurance efforts moving forward!

Best regards,
[Your Name]
================================= Tool Message =================================
Name: send_email

Email sent to qa@company.com - Subject: Reminder: Reviewing New Test Cases
================================== Ai Message ==================================

The reminder email has been successfully s

## Sub-agents as Tools

Wrapping each sub-agent as a tool that the supervisor can invoke creating the layered system in which each sub-agent has a narrow focus with domain-specific tools and prompts. The supervisor will see high-level tools like “schedule_event”, not low-level tools like “create_calendar_event”.

In [11]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """

    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})

    return result["messages"][-1].text      # Returns the result extracted from last message

In [12]:
@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about the meeting')
    """
    
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })

    return result["messages"][-1].text      # Returns the result extracted from last message

## Supervisor Agent
Creates the supervisor that orchestrates the sub-agents. The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual tools/API level.

In [13]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

# Creates supervisor agent
supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)

## Testing
 Tests complete system with complex requests that require coordination across multiple domains.

### Testing over simple single-domain (canledar) request
The supervisor is expected to identify this as a calendar task, call `schedule_event`, and the calendar agent should handle date parsing and event creation.

In [14]:
# MAY TAKE AROUND 1 AND ½ MINUTES TO COMPLETE

query = "Schedule a team standup for tomorrow at 9am"

agent_output = supervisor_agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
    version="v2"
    )

for message in agent_output.value["messages"]:
    message.pretty_print()

================================ Human Message =================================

Schedule a team standup for tomorrow at 9am
================================== Ai Message ==================================
Tool Calls:
  schedule_event (b6704027-ca85-445e-83f5-86baf87ef074)
 Call ID: b6704027-ca85-445e-83f5-86baf87ef074
  Args:
    request: schedule a team standup for tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

The team standup has been successfully scheduled for tomorrow at 9:00 AM.

**Event Details:**
- **Title:** Team Standup
- **Date:** June 23, 2026
- **Time:** 09:00 - 09:30 AM
- **Location:** Meeting Room A
================================== Ai Message ==================================

The team standup has been scheduled for tomorrow at 9:00 AM for 30 minutes in Meeting Room A.


### Testing over multiple-domain (calender + email) request
The supervisor should recognize that this requires both calendar and email actions, call `schedule_event` for the meeting, then call `manage_email` for the reminder. After each sub-agent completes its task, and the supervisor should synthesize both results into a coherent response.

In [15]:
# MAY TAKE AROUND 3 AND ½ MINUTES TO COMPLETE

query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

agent_output = supervisor_agent.invoke(
    {"messages": [{"role": "user", "content": query}]}, 
    stream_mode="values", 
    version="v2"
    )

for message in agent_output.value["messages"]:
    message.pretty_print()

================================ Human Message =================================

Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.
================================== Ai Message ==================================
Tool Calls:
  schedule_event (b65db44f-8b99-44a9-ba40-d81cf0aa8d67)
 Call ID: b65db44f-8b99-44a9-ba40-d81cf0aa8d67
  Args:
    request: meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (7b1b6b53-95c5-420e-8e27-518b157ffcf7)
 Call ID: 7b1b6b53-95c5-420e-8e27-518b157ffcf7
  Args:
    request: send them a reminder about reviewing the new mockups
================================= Tool Message =================================
Name: schedule_event

I've scheduled the meeting for next Tuesday at 2pm.

The meeting for "Meeting with the design team" has been created for June 23, 2026, from 4:00 PM to 5:00 PM (1 hour).

I also checked the calendar for availability on that da